In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                             roc_auc_score, confusion_matrix, classification_report, roc_curve)
import warnings
warnings.filterwarnings('ignore')

In [2]:
#data preparation
df = pd.read_csv('churnml.csv')
print("Shape:", df.shape)
print("\nColumns and Data Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())
print("\nFirst 5 rows:")
print(df.head())
print("\nSummary Statistics (Numeric):")
print(df.describe())
print("\nSummary Statistics (Categorical):")
print(df.describe(include=['object', 'bool']))

Shape: (7043, 21)

Columns and Data Types:
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

Missing Values:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract

In [3]:
# 1. Data Cleaning
df = pd.read_csv('churnml.csv')
blank_total_charges = df[df['TotalCharges'].str.strip() == '']
print("Rows with blank TotalCharges:", len(blank_total_charges))
print(blank_total_charges[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']])

# Convert TotalCharges to numeric, filling blanks (where tenure == 0) with 0.0
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].str.strip(), errors='coerce').fillna(0.0)
df['Churn_Binary'] = (df['Churn'] == 'Yes').astype(int)

print("\nUpdated Numeric Summary:")
print(df[['tenure', 'MonthlyCharges', 'TotalCharges']].describe())

print("\nOverall Churn Distribution:")
print(df['Churn'].value_counts())
print(df['Churn'].value_counts(normalize=True) * 100)

# Groupby stats by Churn
print("\nNumeric Features by Churn:")
print(df.groupby('Churn')[['tenure', 'MonthlyCharges', 'TotalCharges']].agg(['count', 'mean', 'median', 'std', 'min', 'max']))

# Key categorical churn rates
cat_cols = ['Contract', 'InternetService', 'PaymentMethod', 'OnlineSecurity', 'TechSupport', 'SeniorCitizen', 'Partner', 'Dependents', 'PaperlessBilling']
for col in cat_cols:
    print(f"\nChurn Rate by {col}:")
    ct = pd.crosstab(df[col], df['Churn'], normalize='index') * 100
    ct['Count'] = df[col].value_counts()
    print(ct[['Count', 'Yes', 'No']])

Rows with blank TotalCharges: 11
      customerID  tenure  MonthlyCharges TotalCharges Churn
488   4472-LVYGI       0           52.55                 No
753   3115-CZMZD       0           20.25                 No
936   5709-LVOEQ       0           80.85                 No
1082  4367-NUYAO       0           25.75                 No
1340  1371-DWPAZ       0           56.05                 No
3331  7644-OMVMY       0           19.85                 No
3826  3213-VVOLG       0           25.35                 No
4380  2520-SGTTA       0           20.00                 No
5218  2923-ARZLG       0           19.70                 No
6670  4075-WKNIU       0           73.35                 No
6754  2775-SEFEE       0           61.90                 No

Updated Numeric Summary:
            tenure  MonthlyCharges  TotalCharges
count  7043.000000     7043.000000   7043.000000
mean     32.371149       64.761692   2279.734304
std      24.559481       30.090047   2266.794470
min       0.000000       

In [8]:
#EDA & Models 
#Set style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'sans-serif'

df = pd.read_csv('churnml.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].str.strip(), errors='coerce').fillna(0.0)
df['Churn_Binary'] = (df['Churn'] == 'Yes').astype(int)

# Feature Engineering
# 1. Number of value-added services subscribed
services = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 
            'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df['TotalServices'] = (df[services] == 'Yes').sum(axis=1)

# 2. Support/Security Deficit (Has internet but no OnlineSecurity and no TechSupport)
df['Unprotected_Internet'] = ((df['InternetService'] != 'No') & 
                              (df['OnlineSecurity'] == 'No') & 
                              (df['TechSupport'] == 'No')).astype(int)

# 3. Charge-to-Tenure Ratio / Expected vs Actual Total Charges difference
df['Charge_Difference'] = df['TotalCharges'] - (df['tenure'] * df['MonthlyCharges'])
df['Avg_Monthly_Spend'] = np.where(df['tenure'] > 0, df['TotalCharges'] / df['tenure'], df['MonthlyCharges'])

# 4. Tenure Cohort
df['Tenure_Group'] = pd.cut(df['tenure'], bins=[-1, 12, 24, 48, 60, 72], 
                            labels=['0-1 Yr', '1-2 Yrs', '2-4 Yrs', '4-5 Yrs', '5-6 Yrs'])

print("Churn Rate by Tenure Group:")
print(pd.crosstab(df['Tenure_Group'], df['Churn'], normalize='index') * 100)
print("\nChurn Rate by Unprotected_Internet:")
print(pd.crosstab(df['Unprotected_Internet'], df['Churn'], normalize='index') * 100)

# Figure 1: EDA Dashboard
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle('Exploratory Data Analysis: Customer Churn Drivers & Distributions', fontsize=16, fontweight='bold', y=0.98)

# 1A: Tenure vs Churn KDE/Histogram
sns.histplot(data=df, x='tenure', hue='Churn', multiple='stack', palette={'No': '#2b5c8f', 'Yes': '#d95f02'}, bins=36, ax=axes[0, 0])
axes[0, 0].set_title('Customer Tenure Distribution by Churn Status', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('Tenure (Months)')
axes[0, 0].set_ylabel('Number of Customers')

# 1B: Monthly Charges Distribution by Churn
sns.kdeplot(data=df, x='MonthlyCharges', hue='Churn', fill=True, common_norm=False, palette={'No': '#2b5c8f', 'Yes': '#d95f02'}, alpha=0.5, ax=axes[0, 1])
axes[0, 1].set_title('Monthly Charges Density by Churn Status', fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('Monthly Charges ($)')
axes[0, 1].set_ylabel('Density')

# 1C: Churn Rate by Contract & Internet Service
contract_internet = df.groupby(['Contract', 'InternetService'])['Churn_Binary'].mean().unstack() * 100
contract_internet.plot(kind='bar', ax=axes[1, 0], color=['#388e3c', '#d95f02', '#757575'], rot=0)
axes[1, 0].set_title('Churn Rate (%) by Contract Type & Internet Service', fontsize=13, fontweight='bold')
axes[1, 0].set_xlabel('Contract Type')
axes[1, 0].set_ylabel('Churn Rate (%)')
axes[1, 0].legend(title='Internet Service')
for container in axes[1, 0].containers:
    axes[1, 0].bar_label(container, fmt='%.1f%%', padding=3, fontsize=9)
axes[1, 0].set_ylim(0, 65)

# 1D: Churn Rate by Payment Method
pm_churn = df.groupby('PaymentMethod')['Churn_Binary'].mean().sort_values(ascending=True) * 100
pm_churn.plot(kind='barh', ax=axes[1, 1], color=['#2b5c8f', '#2b5c8f', '#2b5c8f', '#d95f02'])
axes[1, 1].set_title('Churn Rate (%) by Payment Method', fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel('Churn Rate (%)')
axes[1, 1].set_ylabel('')
for container in axes[1, 1].containers:
    axes[1, 1].bar_label(container, fmt='%.1f%%', padding=3, fontsize=10)
axes[1, 1].set_xlim(0, 55)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('eda_churn_analysis.png', dpi=300)
plt.close()

Churn Rate by Tenure Group:
Churn                No        Yes
Tenure_Group                      
0-1 Yr        52.561757  47.438243
1-2 Yrs       71.289062  28.710938
2-4 Yrs       79.611041  20.388959
4-5 Yrs       85.576923  14.423077
5-6 Yrs       93.390192   6.609808

Churn Rate by Unprotected_Internet:
Churn                        No        Yes
Unprotected_Internet                      
0                     86.213808  13.786192
1                     51.037995  48.962005


In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    roc_curve
)

# ---------------------------------------------------------
# 1. Set Plot Style & Load Data
# ---------------------------------------------------------
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('seaborn-whitegrid')
plt.rcParams['font.family'] = 'sans-serif'

df = pd.read_csv('churnml.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].str.strip(), errors='coerce').fillna(0.0)
df['Churn_Binary'] = (df['Churn'] == 'Yes').astype(int)

# ---------------------------------------------------------
# 2. Feature Engineering
# ---------------------------------------------------------
# 2.1 Number of value-added services subscribed
services = [
    'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'
]
df['TotalServices'] = (df[services] == 'Yes').sum(axis=1)

# 2.2 Support/Security Deficit (Has internet but no OnlineSecurity and no TechSupport)
df['Unprotected_Internet'] = (
    (df['InternetService'] != 'No') &
    (df['OnlineSecurity'] == 'No') &
    (df['TechSupport'] == 'No')
).astype(int)

# 2.3 Charge-to-Tenure Ratio / Expected vs Actual Total Charges difference
df['Charge_Difference'] = df['TotalCharges'] - (df['tenure'] * df['MonthlyCharges'])
df['Avg_Monthly_Spend'] = np.where(
    df['tenure'] > 0,
    df['TotalCharges'] / df['tenure'],
    df['MonthlyCharges']
)

# 2.4 Tenure Cohort
df['Tenure_Group'] = pd.cut(
    df['tenure'],
    bins=[-1, 12, 24, 48, 60, 72],
    labels=['0-1 Yr', '1-2 Yrs', '2-4 Yrs', '4-5 Yrs', '5-6 Yrs']
)

print("Churn Rate by Tenure Group:")
print(pd.crosstab(df['Tenure_Group'], df['Churn'], normalize='index') * 100)

print("\nChurn Rate by Unprotected_Internet:")
print(pd.crosstab(df['Unprotected_Internet'], df['Churn'], normalize='index') * 100)

# ---------------------------------------------------------
# 3. Figure 1: EDA Dashboard
# ---------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle(
    'Exploratory Data Analysis: Customer Churn Drivers & Distributions',
    fontsize=16, fontweight='bold', y=0.98
)

# 1A: Tenure vs Churn Histogram
sns.histplot(
    data=df, x='tenure', hue='Churn', multiple='stack',
    palette={'No': '#2b5c8f', 'Yes': '#d95f02'}, bins=36, ax=axes[0, 0]
)
axes[0, 0].set_title('Customer Tenure Distribution by Churn Status', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('Tenure (Months)')
axes[0, 0].set_ylabel('Number of Customers')

# 1B: Monthly Charges Distribution by Churn
sns.kdeplot(
    data=df, x='MonthlyCharges', hue='Churn', fill=True, common_norm=False,
    palette={'No': '#2b5c8f', 'Yes': '#d95f02'}, alpha=0.5, ax=axes[0, 1]
)
axes[0, 1].set_title('Monthly Charges Density by Churn Status', fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('Monthly Charges ($)')
axes[0, 1].set_ylabel('Density')

# 1C: Churn Rate by Contract & Internet Service
contract_internet = df.groupby(['Contract', 'InternetService'])['Churn_Binary'].mean().unstack() * 100
contract_internet.plot(kind='bar', ax=axes[1, 0], color=['#388e3c', '#d95f02', '#757575'], rot=0)
axes[1, 0].set_title('Churn Rate (%) by Contract Type & Internet Service', fontsize=13, fontweight='bold')
axes[1, 0].set_xlabel('Contract Type')
axes[1, 0].set_ylabel('Churn Rate (%)')
axes[1, 0].legend(title='Internet Service')

for container in axes[1, 0].containers:
    axes[1, 0].bar_label(container, fmt='%.1f%%', padding=3, fontsize=9)
axes[1, 0].set_ylim(0, 65)

# 1D: Churn Rate by Payment Method
pm_churn = df.groupby('PaymentMethod')['Churn_Binary'].mean().sort_values(ascending=True) * 100
pm_churn.plot(kind='barh', ax=axes[1, 1], color=['#2b5c8f', '#2b5c8f', '#2b5c8f', '#d95f02'])
axes[1, 1].set_title('Churn Rate (%) by Payment Method', fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel('Churn Rate (%)')
axes[1, 1].set_ylabel('')

for container in axes[1, 1].containers:
    axes[1, 1].bar_label(container, fmt='%.1f%%', padding=3, fontsize=10)
axes[1, 1].set_xlim(0, 55)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('eda_churn_analysis.png', dpi=300)
plt.close()

# ---------------------------------------------------------
# 4. Predictive Analytics Pipeline
# ---------------------------------------------------------
feature_cols = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
    'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
    'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
    'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
    'MonthlyCharges', 'TotalCharges', 'TotalServices', 'Unprotected_Internet',
    'Charge_Difference', 'Avg_Monthly_Spend'
]

X = df[feature_cols].copy()
y = df['Churn_Binary']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

num_cols = [
    'tenure', 'MonthlyCharges', 'TotalCharges',
    'TotalServices', 'Charge_Difference', 'Avg_Monthly_Spend'
]
cat_cols_model = [c for c in feature_cols if c not in num_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols_model)
    ]
)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Logistic Regression (Balanced)': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=10, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, max_depth=4, random_state=42)
}

results = []
roc_data = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    pipe = Pipeline([('prep', preprocessor), ('clf', model)])
    cv_scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=['roc_auc', 'f1', 'accuracy'])
    
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)
    pr_auc = average_precision_score(y_test, y_prob)
    
    results.append({
        'Model': name,
        'CV ROC-AUC': f"{cv_scores['test_roc_auc'].mean():.4f} ± {cv_scores['test_roc_auc'].std():.4f}",
        'Test Accuracy': acc,
        'Test Precision': prec,
        'Test Recall': rec,
        'Test F1-Score': f1,
        'Test ROC-AUC': roc_auc,
        'Test PR-AUC': pr_auc
    })
    
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_data[name] = (fpr, tpr, roc_auc, pipe, y_prob)

results_df = pd.DataFrame(results)
print("\nModel Evaluation Summary:")
print(results_df.to_string(index=False))

# ---------------------------------------------------------
# 5. Feature Importance & Threshold Optimization
# ---------------------------------------------------------
gb_pipe = roc_data['Gradient Boosting'][3]
ohe_feature_names = gb_pipe.named_steps['prep'].named_transformers_['cat'].get_feature_names_out(cat_cols_model)
all_feature_names = num_cols + list(ohe_feature_names)

gb_importances = pd.Series(
    gb_pipe.named_steps['clf'].feature_importances_,
    index=all_feature_names
).sort_values(ascending=False)
print("\nTop 10 Gradient Boosting Feature Importances:")
print(gb_importances.head(10))

lr_pipe = roc_data['Logistic Regression'][3]
lr_coefs = pd.Series(
    lr_pipe.named_steps['clf'].coef_[0],
    index=all_feature_names
).sort_values(key=abs, ascending=False)
print("\nTop 10 Logistic Regression Coefficients (Log-Odds):")
print(lr_coefs.head(10))

gb_prob = roc_data['Gradient Boosting'][4]
thresholds = np.arange(0.1, 0.9, 0.05)
best_thresh = 0.5
best_f1 = 0.0

for t in thresholds:
    f1_t = f1_score(y_test, (gb_prob >= t).astype(int))
    if f1_t > best_f1:
        best_f1 = f1_t
        best_thresh = t

opt_recall = recall_score(y_test, (gb_prob >= best_thresh).astype(int))
opt_prec = precision_score(y_test, (gb_prob >= best_thresh).astype(int))
print(
    f"\nOptimal Threshold for Gradient Boosting F1: {best_thresh:.2f} -> "
    f"F1: {best_f1:.4f}, Recall: {opt_recall:.4f}, Precision: {opt_prec:.4f}"
)

# ---------------------------------------------------------
# 6. Risk Segmentation on Full Dataset
# ---------------------------------------------------------
full_probs = gb_pipe.predict_proba(X)[:, 1]
df['Churn_Probability'] = full_probs
df['Risk_Segment'] = pd.cut(
    df['Churn_Probability'],
    bins=[0, 0.25, 0.50, 1.0],
    labels=['Low Risk (<25%)', 'Medium Risk (25-50%)', 'High Risk (>50%)']
)

risk_summary = df.groupby('Risk_Segment', observed=False).agg(
    Customer_Count=('customerID', 'count'),
    Actual_Churn_Rate=('Churn_Binary', lambda x: x.mean() * 100),
    Avg_Monthly_Charges=('MonthlyCharges', 'mean'),
    Total_Monthly_Revenue_At_Risk=('MonthlyCharges', 'sum')
)
print("\nRisk Segmentation Summary (Full Dataset):")
print(risk_summary)

# ---------------------------------------------------------
# 7. Figure 2: Predictive Modeling & Feature Importance
# ---------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle(
    'Predictive Analytics: Model Performance & Feature Importance',
    fontsize=16, fontweight='bold', y=0.98
)

# 2A: ROC Curves
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
for (name, (fpr, tpr, auc_val, _, _)), col in zip(roc_data.items(), colors):
    axes[0].plot(fpr, tpr, label=f"{name} (AUC = {auc_val:.4f})", linewidth=2, color=col)

axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Guess (AUC = 0.5000)')
axes[0].set_title('ROC Curve Comparison (Test Set, n = 1,409)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('False Positive Rate (1 - Specificity)')
axes[0].set_ylabel('True Positive Rate (Sensitivity / Recall)')
axes[0].legend(loc='lower right', frameon=True)

# 2B: Top 10 Feature Importances (Gradient Boosting)
top10_gb = gb_importances.head(10).sort_values(ascending=True)
top10_gb.plot(kind='barh', ax=axes[1], color='#2b5c8f')
axes[1].set_title('Top 10 Predictive Features (Gradient Boosting)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Relative Feature Importance')

for container in axes[1].containers:
    axes[1].bar_label(container, fmt='%.3f', padding=3, fontsize=9)
axes[1].set_xlim(0, max(top10_gb) * 1.15)

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig('predictive_model_evaluation.png', dpi=300)
plt.close()

Churn Rate by Tenure Group:
Churn                No        Yes
Tenure_Group                      
0-1 Yr        52.561757  47.438243
1-2 Yrs       71.289062  28.710938
2-4 Yrs       79.611041  20.388959
4-5 Yrs       85.576923  14.423077
5-6 Yrs       93.390192   6.609808

Churn Rate by Unprotected_Internet:
Churn                        No        Yes
Unprotected_Internet                      
0                     86.213808  13.786192
1                     51.037995  48.962005

Model Evaluation Summary:
                         Model      CV ROC-AUC  Test Accuracy  Test Precision  Test Recall  Test F1-Score  Test ROC-AUC  Test PR-AUC
           Logistic Regression 0.8454 ± 0.0130       0.804826        0.656151     0.556150       0.602026      0.842370     0.634003
Logistic Regression (Balanced) 0.8452 ± 0.0127       0.738112        0.504318     0.780749       0.612802      0.841908     0.632129
                 Random Forest 0.8464 ± 0.0114       0.804116        0.681481     0.491979  

In [12]:
# Verify universal compatibility across pandas/matplotlib/sklearn versions
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, roc_curve,
    precision_recall_curve, confusion_matrix
)

print("Version check passed.")

# 1. Load & Clean Data
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('seaborn-whitegrid')
plt.rcParams['font.family'] = 'sans-serif'

df = pd.read_csv('churnml.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].str.strip(), errors='coerce').fillna(0.0)
df['Churn_Binary'] = (df['Churn'] == 'Yes').astype(int)

# 2. Feature Engineering
services = [
    'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'
]
df['TotalServices'] = (df[services] == 'Yes').sum(axis=1)
df['Unprotected_Internet'] = (
    (df['InternetService'] != 'No') &
    (df['OnlineSecurity'] == 'No') &
    (df['TechSupport'] == 'No')
).astype(int)
df['Charge_Difference'] = df['TotalCharges'] - (df['tenure'] * df['MonthlyCharges'])
df['Avg_Monthly_Spend'] = np.where(df['tenure'] > 0, df['TotalCharges'] / df['tenure'], df['MonthlyCharges'])
df['Tenure_Group'] = pd.cut(
    df['tenure'],
    bins=[-1, 12, 24, 48, 60, 72],
    labels=['0-1 Yr', '1-2 Yrs', '2-4 Yrs', '4-5 Yrs', '5-6 Yrs']
)

# ---------------------------------------------------------
# DASHBOARD 1: COMPREHENSIVE 6-PANEL EDA DASHBOARD
# ---------------------------------------------------------
fig, axes = plt.subplots(3, 2, figsize=(16, 15))
fig.suptitle('Executive EDA Dashboard: Customer Churn Behavior & Revenue Drivers (n = 7,043)', fontsize=18, fontweight='bold', y=0.98)

# Panel 1: Overall Churn & Revenue Breakdown (Donut / Bar Combo)
churn_counts = df['Churn'].value_counts()
colors_pie = ['#2b5c8f', '#d95f02']
wedges, texts, autotexts = axes[0, 0].pie(
    churn_counts, labels=['Retained (No)', 'Churned (Yes)'], autopct='%1.1f%%',
    startangle=140, colors=colors_pie, wedgeprops=dict(width=0.45, edgecolor='white'),
    textprops=dict(fontsize=11, fontweight='bold')
)
for autotext in autotexts:
    autotext.set_color('white')
axes[0, 0].set_title('Overall Customer Retention vs. Churn Split\n(1,869 Churned = $139.1K Lost MRR)', fontsize=13, fontweight='bold')

# Panel 2: Tenure Cohort Churn Rate & Volume
tenure_ct = df.groupby('Tenure_Group', observed=False)['Churn_Binary'].agg(['count', 'mean'])
tenure_ct['mean'] *= 100
ax2 = axes[0, 1]
bars = ax2.bar(tenure_ct.index, tenure_ct['mean'], color=['#d95f02', '#e68a00', '#5c85d6', '#3366cc', '#2b5c8f'], edgecolor='black', alpha=0.85)
ax2.set_title('Churn Rate (%) Across Customer Tenure Cohorts', fontsize=13, fontweight='bold')
ax2.set_xlabel('Tenure Cohort')
ax2.set_ylabel('Churn Rate (%)')
ax2.set_ylim(0, 58)
for bar, (_, row) in zip(bars, tenure_ct.iterrows()):
    ax2.annotate(f"{row['mean']:.1f}%\n(n={int(row['count'])})",
                 xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                 xytext=(0, 4), textcoords="offset points", ha='center', va='bottom', fontsize=9.5, fontweight='bold')

# Panel 3: Monthly Charges KDE by Churn Status
sns.kdeplot(data=df, x='MonthlyCharges', hue='Churn', fill=True, common_norm=False,
            palette={'No': '#2b5c8f', 'Yes': '#d95f02'}, alpha=0.45, linewidth=2, ax=axes[1, 0])
axes[1, 0].axvline(df[df['Churn']=='Yes']['MonthlyCharges'].median(), color='#d95f02', linestyle='--', linewidth=1.8, label='Churned Median ($79.65)')
axes[1, 0].axvline(df[df['Churn']=='No']['MonthlyCharges'].median(), color='#2b5c8f', linestyle='--', linewidth=1.8, label='Retained Median ($64.43)')
axes[1, 0].set_title('Monthly Charges Distribution & Median Shift', fontsize=13, fontweight='bold')
axes[1, 0].set_xlabel('Monthly Charges ($)')
axes[1, 0].set_ylabel('Density')
axes[1, 0].legend(loc='upper left', frameon=True)

# Panel 4: Contract Type x Internet Service Churn Rate
contract_internet = df.groupby(['Contract', 'InternetService'])['Churn_Binary'].mean().unstack() * 100
contract_internet.plot(kind='bar', ax=axes[1, 1], color=['#388e3c', '#d95f02', '#757575'], rot=0, edgecolor='black')
axes[1, 1].set_title('Churn Rate (%) by Contract Type & Internet Tier', fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel('Contract Type')
axes[1, 1].set_ylabel('Churn Rate (%)')
axes[1, 1].legend(title='Internet Service', frameon=True)
for container in axes[1, 1].containers:
    axes[1, 1].bar_label(container, fmt='%.1f%%', padding=3, fontsize=9, fontweight='bold')
axes[1, 1].set_ylim(0, 65)

# Panel 5: Payment Method Churn Rate
pm_churn = df.groupby('PaymentMethod')['Churn_Binary'].mean().sort_values(ascending=True) * 100
pm_churn.plot(kind='barh', ax=axes[2, 0], color=['#2b5c8f', '#2b5c8f', '#2b5c8f', '#d95f02'], edgecolor='black')
axes[2, 0].set_title('Churn Rate (%) by Payment Method', fontsize=13, fontweight='bold')
axes[2, 0].set_xlabel('Churn Rate (%)')
axes[2, 0].set_ylabel('')
for container in axes[2, 0].containers:
    axes[2, 0].bar_label(container, fmt='%.1f%%', padding=4, fontsize=10, fontweight='bold')
axes[2, 0].set_xlim(0, 55)

# Panel 6: Impact of Unprotected Internet (No Security & No Tech Support) & Senior Citizen
risk_factors = pd.DataFrame({
    'Group': ['Protected / No Internet', 'Unprotected Internet\n(No Sec & No Tech)', 'Non-Senior Citizen', 'Senior Citizen'],
    'Churn_Rate': [
        df[df['Unprotected_Internet']==0]['Churn_Binary'].mean()*100,
        df[df['Unprotected_Internet']==1]['Churn_Binary'].mean()*100,
        df[df['SeniorCitizen']==0]['Churn_Binary'].mean()*100,
        df[df['SeniorCitizen']==1]['Churn_Binary'].mean()*100
    ]
})
bars6 = axes[2, 1].bar(risk_factors['Group'], risk_factors['Churn_Rate'], color=['#2b5c8f', '#d95f02', '#2b5c8f', '#e68a00'], edgecolor='black')
axes[2, 1].set_title('Support Deficit & Demographic Vulnerability (Churn %)', fontsize=13, fontweight='bold')
axes[2, 1].set_ylabel('Churn Rate (%)')
axes[2, 1].set_ylim(0, 60)
for bar in bars6:
    axes[2, 1].annotate(f"{bar.get_height():.1f}%",
                        xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                        xytext=(0, 4), textcoords="offset points", ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('dashboard_1_eda.png', dpi=300)
plt.close()

# ---------------------------------------------------------
# DASHBOARD 2: PREDICTIVE MODELS & RISK SEGMENTATION DASHBOARD
# ---------------------------------------------------------
feature_cols = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
    'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
    'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
    'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
    'MonthlyCharges', 'TotalCharges', 'TotalServices', 'Unprotected_Internet',
    'Charge_Difference', 'Avg_Monthly_Spend'
]

X = df[feature_cols].copy()
y = df['Churn_Binary']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'TotalServices', 'Charge_Difference', 'Avg_Monthly_Spend']
cat_cols_model = [c for c in feature_cols if c not in num_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols_model)
    ]
)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Logistic Regression (Balanced)': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=10, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, max_depth=4, random_state=42)
}

roc_data = {}
pr_data = {}
for name, model in models.items():
    pipe = Pipeline([('prep', preprocessor), ('clf', model)])
    pipe.fit(X_train, y_train)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    prec_curve, rec_curve, _ = precision_recall_curve(y_test, y_prob)
    roc_data[name] = (fpr, tpr, roc_auc_score(y_test, y_prob), pipe, y_prob)
    pr_data[name] = (rec_curve, prec_curve, average_precision_score(y_test, y_prob))

gb_pipe = roc_data['Gradient Boosting'][3]
gb_prob = roc_data['Gradient Boosting'][4]
ohe_feature_names = gb_pipe.named_steps['prep'].named_transformers_['cat'].get_feature_names_out(cat_cols_model)
all_feature_names = num_cols + list(ohe_feature_names)
gb_importances = pd.Series(gb_pipe.named_steps['clf'].feature_importances_, index=all_feature_names).sort_values(ascending=False)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Predictive Analytics Dashboard: Model Evaluation, Threshold Tuning & Risk Tiers', fontsize=18, fontweight='bold', y=0.98)

# Panel 1: ROC Curves & PR-AUC Summary
colors_m = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
for (name, (fpr, tpr, auc_val, _, _)), col in zip(roc_data.items(), colors_m):
    pr_auc_val = pr_data[name][2]
    axes[0, 0].plot(fpr, tpr, label=f"{name} (ROC-AUC={auc_val:.3f} | PR-AUC={pr_auc_val:.3f})", linewidth=2.2, color=col)
axes[0, 0].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Baseline (AUC=0.500)')
axes[0, 0].set_title('ROC Curve Comparison on Holdout Test Set (n = 1,409)', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('False Positive Rate (1 - Specificity)')
axes[0, 0].set_ylabel('True Positive Rate (Recall)')
axes[0, 0].legend(loc='lower right', frameon=True, fontsize=9.5)

# Panel 2: Top 10 Feature Importances (Gradient Boosting)
top10_gb = gb_importances.head(10).sort_values(ascending=True)
top10_gb.plot(kind='barh', ax=axes[0, 1], color='#2b5c8f', edgecolor='black')
axes[0, 1].set_title('Top 10 Predictive Features (Gradient Boosting)', fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('Relative Feature Importance')
for container in axes[0, 1].containers:
    axes[0, 1].bar_label(container, fmt='%.3f', padding=4, fontsize=9.5, fontweight='bold')
axes[0, 1].set_xlim(0, max(top10_gb) * 1.18)

# Panel 3: Precision, Recall & F1 Trade-off Curve across Thresholds (Gradient Boosting)
thresholds = np.linspace(0.05, 0.90, 50)
precs, recs, f1s = [], [], []
for t in thresholds:
    preds_t = (gb_prob >= t).astype(int)
    precs.append(precision_score(y_test, preds_t, zero_division=0))
    recs.append(recall_score(y_test, preds_t))
    f1s.append(f1_score(y_test, preds_t))

axes[1, 0].plot(thresholds, recs, label='Recall (Sensitivity)', color='#d95f02', linewidth=2.2)
axes[1, 0].plot(thresholds, precs, label='Precision', color='#2b5c8f', linewidth=2.2)
axes[1, 0].plot(thresholds, f1s, label='F1-Score', color='#2ca02c', linewidth=2.5)
axes[1, 0].axvline(0.30, color='red', linestyle='--', linewidth=1.8, label='Optimal Threshold (p = 0.30, F1 = 0.622)')
axes[1, 0].axvline(0.50, color='gray', linestyle=':', linewidth=1.5, label='Default Threshold (p = 0.50, F1 = 0.580)')
axes[1, 0].set_title('Decision Threshold Optimization (Gradient Boosting)', fontsize=13, fontweight='bold')
axes[1, 0].set_xlabel('Predicted Churn Probability Threshold (p)')
axes[1, 0].set_ylabel('Metric Score')
axes[1, 0].legend(loc='lower left', frameon=True, fontsize=9.5)
axes[1, 0].set_ylim(0, 1.05)

# Panel 4: Confusion Matrix at Optimal Threshold (p = 0.30) vs Revenue Risk Segmentation
cm = confusion_matrix(y_test, (gb_prob >= 0.30).astype(int))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[1, 1],
            xticklabels=['Predicted Retained (0)', 'Predicted Churn (1)'],
            yticklabels=['Actual Retained (0)', 'Actual Churn (1)'],
            annot_kws={'size': 14, 'weight': 'bold'})
axes[1, 1].set_title('Test Set Confusion Matrix at Optimal Threshold (p = 0.30)\nCaptures 280 / 374 Churners (74.9% Recall)', fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel('Model Prediction')
axes[1, 1].set_ylabel('Actual Outcome')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('dashboard_2_models.png', dpi=300)
plt.close()

# ---------------------------------------------------------
# DASHBOARD 3: CUSTOMER RISK SEGMENTATION & MRR EXPOSURE
# ---------------------------------------------------------
full_probs = gb_pipe.predict_proba(X)[:, 1]
df['Churn_Probability'] = full_probs
df['Risk_Segment'] = pd.cut(
    df['Churn_Probability'],
    bins=[0, 0.25, 0.50, 1.0],
    labels=['Low Risk (<25%)', 'Medium Risk (25-50%)', 'High Risk (>50%)']
)

risk_summary = df.groupby('Risk_Segment', observed=False).agg(
    Customer_Count=('customerID', 'count'),
    Actual_Churn_Rate=('Churn_Binary', lambda x: x.mean() * 100),
    Total_MRR=('MonthlyCharges', lambda x: x.sum() / 1000.0) # in $K
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
fig.suptitle('Strategic Risk Segmentation: Customer Volume, Actual Churn & Monthly Revenue at Risk', fontsize=16, fontweight='bold', y=0.99)

# Left: Customer Count & Actual Churn Rate
ax_l = axes[0]
b1 = ax_l.bar(risk_summary.index, risk_summary['Customer_Count'], color=['#2b5c8f', '#e68a00', '#d95f02'], edgecolor='black', width=0.55)
ax_l.set_title('Customer Volume & Actual Churn Rate by Risk Tier', fontsize=13, fontweight='bold')
ax_l.set_ylabel('Number of Customers')
ax_l.set_ylim(0, 4800)
for bar, (_, row) in zip(b1, risk_summary.iterrows()):
    ax_l.annotate(f"{int(row['Customer_Count']):,} users\n({row['Actual_Churn_Rate']:.1f}% Churn)",
                  xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                  xytext=(0, 4), textcoords="offset points", ha='center', va='bottom', fontsize=10, fontweight='bold')

# Right: Monthly Recurring Revenue (MRR in $K)
ax_r = axes[1]
b2 = ax_r.bar(risk_summary.index, risk_summary['Total_MRR'], color=['#2b5c8f', '#e68a00', '#d95f02'], edgecolor='black', width=0.55)
ax_r.set_title('Monthly Recurring Revenue (MRR) Exposure by Tier ($K)', fontsize=13, fontweight='bold')
ax_r.set_ylabel('Total Monthly Revenue ($ Thousands)')
ax_r.set_ylim(0, 280)
for bar, (_, row) in zip(b2, risk_summary.iterrows()):
    ax_r.annotate(f"${row['Total_MRR']:.1f}K / mo\n(${row['Total_MRR']*12:.0f}K / yr)",
                  xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                  xytext=(0, 4), textcoords="offset points", ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig('dashboard_3_risk_tiers.png', dpi=300)
plt.close()
print("All 3 dashboards generated successfully.")

Version check passed.
All 3 dashboards generated successfully.
